
# Exp4 — Multi-Probe Predictive Utility: Ridge vs. MLP

> *Is the paper's controlled predictive utility \(P\) merely an artifact of the Ridge probe?*

This experiment defines predictive utility **independently under two controlled probes**:

\[
P^{\mathrm{Ridge}}_{ij}
=
100\frac{L_i^{\mathrm{Ridge}}-L_{i\leftarrow j}^{\mathrm{Ridge}}}
{L_i^{\mathrm{Ridge}}},
\qquad
P^{\mathrm{MLP}}_{ij}
=
100\frac{L_i^{\mathrm{MLP}}-L_{i\leftarrow j}^{\mathrm{MLP}}}
{L_i^{\mathrm{MLP}}}.
\]

Both probes use the **same chronological fit/score split**, target-history features, source features, and source-candidate pool.  
Source summary features are residualized against target-history summary features using **fit data only**, matching the controlled protocol.

The MLP utility is **not** the previous Exp2 result. Exp2 kept Ridge-selected source sets and only changed the final controlled forecaster. Here the MLP itself independently scores each candidate source.

### Pre-registered primary questions

1. Do \(P^{\mathrm{Ridge}}\) and \(P^{\mathrm{MLP}}\) produce similar source rankings?
2. On the existing matched EPI conditions, does \(P^{\mathrm{MLP}}\) align with neural functional reliance any better than \(P^{\mathrm{Ridge}}\)?
3. If Ridge–MLP agreement is low, report that honestly as evidence that predictive utility itself is probe-conditioned.

### Run modes

- `screen`: the 8 existing EPI diagnostic conditions. **Runtime/debug only; not the final General-20 claim.**
- `full`: all 20 general benchmark conditions (5 datasets × 4 horizons).

Run the screen first to verify runtime and paths. Then set `RUN_MODE="full"` and rerun **regardless of the screen outcome**.


In [1]:

from pathlib import Path
import gc, math, random, time, warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.4.1+cu121 | device: cuda
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


### v2 compatibility fix

`safe_spearman()` now supports both older and newer SciPy return objects:
`SpearmanrResult.correlation`, `SignificanceResult.statistic`, or tuple-style `[0]`.

This change affects only result aggregation; it does **not** change the experiment or saved pair-level utilities.


## 1. Configuration

In [2]:

PROJECT_ROOT_CANDIDATES = [
    Path("/data/code/2026_08"),      # Rethinking project server
    Path("/data/code/rethinking_cross_channel"),
]
PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if p.exists()), PROJECT_ROOT_CANDIDATES[0])

OUTPUT_DIR = PROJECT_ROOT / "results_multiprobe_predictive_utility"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 96
GENERAL_HORIZONS = [96, 192, 336, 720]

MAX_TOPK = 10
MAX_TARGET_CHANNELS = 32
MAX_SOURCE_CANDIDATES = 128
MAX_TRAIN_ORIGINS = 5000
TEACHER_FIT_FRACTION = 0.70

RIDGE_LAMBDA = 1e-3
SOURCE_RIDGE_LAMBDA = 1e-2

# MLP source probes: same conceptual two-hidden-layer controlled MLP family as Exp2.
MLP_HIDDEN = 128
MLP_EPOCHS = 30
MLP_BATCH_SIZE = 256
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-4
MLP_DROPOUT = 0.1

# Start with screen only to validate runtime/path fidelity.
# After successful execution, change to "full" and rerun regardless of result direction.
RUN_MODE = "screen"  # "screen" or "full"

SCREEN_CONDITIONS = [
    ("Electricity", 192),
    ("Electricity", 336),
    ("Solar", 336),
    ("Solar", 720),
    ("Weather", 720),
    ("ETTh1", 336),
    ("ETTh1", 720),
    ("ETTm1", 720),
]

FULL_CONDITIONS = [
    (d, h)
    for d in ["Electricity", "Weather", "Solar", "ETTh1", "ETTm1"]
    for h in GENERAL_HORIZONS
]

# One seed is enough for the screen/runtime check.
# Full mode uses three seeds and averages MLP utilities across seeds.
SCREEN_MLP_SEEDS = [2026]
FULL_MLP_SEEDS = [2026, 2027, 2028]

# Existing EPI pair-level file from the Rethinking pipeline.
EPI_PAIR_FILE = (
    PROJECT_ROOT
    / "results_dependency_predictive_neural_importance"
    / "pair_scores_all.csv"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("EPI pair file exists:", EPI_PAIR_FILE.exists())


PROJECT_ROOT: /data/code/2026_08
OUTPUT_DIR: /data/code/2026_08/results_multiprobe_predictive_utility
RUN_MODE: screen
EPI pair file exists: True


## 2. Dataset protocol — matched to the existing controlled experiment

In [3]:

DATASET_REGISTRY = {
    "Electricity": {
        "paths": [
            "/data/dataset/electricity/electricity.csv",
            "/data/dataset/ECL/electricity.csv",
            "/data/dataset/electricity.csv",
        ],
    },
    "Weather": {
        "paths": [
            "/data/dataset/weather/weather.csv",
            "/data/dataset/weather.csv",
        ],
    },
    "Solar": {
        "paths": [
            "/data/dataset/solar/solar_AL.txt",
            "/data/dataset/Solar/solar_AL.txt",
            "/data/dataset/solar_AL.txt",
        ],
    },
    "ETTh1": {
        "paths": [
            "/data/dataset/ETT-small/ETTh1.csv",
            "/data/dataset/ETTh1.csv",
        ],
    },
    "ETTm1": {
        "paths": [
            "/data/dataset/ETT-small/ETTm1.csv",
            "/data/dataset/ETTm1.csv",
        ],
    },
}

def resolve_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

rows = []
for name, spec in DATASET_REGISTRY.items():
    p = resolve_path(spec["paths"])
    rows.append({"dataset": name, "path": None if p is None else str(p), "found": p is not None})

path_df = pd.DataFrame(rows)
display(path_df)

if not path_df["found"].all():
    print("\nWARNING: Edit DATASET_REGISTRY paths before running the main experiment.")


,dataset,path,found
0,Electricity,/data/dataset/electricity/electricity.csv,True
1,Weather,/data/dataset/weather/weather.csv,True
2,Solar,/data/dataset/solar/solar_AL.txt,True
3,ETTh1,/data/dataset/ETT-small/ETTh1.csv,True
4,ETTm1,/data/dataset/ETT-small/ETTm1.csv,True


In [4]:

def load_series(dataset_name, path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        date_col = None
        for candidate in ["date", "datetime", "timestamp", "time"]:
            if candidate in df.columns:
                date_col = candidate
                break
        if date_col is None:
            first = df.columns[0]
            if not pd.api.types.is_numeric_dtype(df[first]):
                date_col = first
        if date_col is not None:
            df = df.drop(columns=[date_col])
        df = df.select_dtypes(include=[np.number])
        x = df.to_numpy(dtype=np.float32)
    else:
        try:
            x = np.loadtxt(path, delimiter=",", dtype=np.float32)
        except Exception:
            x = np.loadtxt(path, dtype=np.float32)
        if x.ndim == 1:
            x = x[:, None]

    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError(f"{dataset_name}: invalid shape/content {x.shape}")
    return x

def split_boundaries(dataset_name, T):
    if dataset_name == "ETTh1":
        tr = 12 * 30 * 24
        va = tr + 4 * 30 * 24
        te = va + 4 * 30 * 24
        return min(tr, T), min(va, T), min(te, T)
    if dataset_name == "ETTm1":
        unit = 30 * 24 * 4
        tr = 12 * unit
        va = tr + 4 * unit
        te = va + 4 * unit
        return min(tr, T), min(va, T), min(te, T)
    return int(0.70 * T), int(0.80 * T), T

def standardize_train_only(x, train_end):
    mu = x[:train_end].mean(axis=0, keepdims=True)
    sd = np.maximum(x[:train_end].std(axis=0, keepdims=True), 1e-6)
    return ((x - mu) / sd).astype(np.float32)

def evenly_subsample(values, max_n):
    values = np.asarray(values, dtype=np.int64)
    if len(values) <= max_n:
        return values
    idx = np.linspace(0, len(values)-1, max_n, dtype=np.int64)
    return values[idx]

def make_train_origins(train_end):
    last = int(train_end) - max(GENERAL_HORIZONS)
    if last < SEQ_LEN:
        raise RuntimeError("No valid training origins.")
    origins = np.arange(SEQ_LEN, last + 1, dtype=np.int64)
    return evenly_subsample(origins, MAX_TRAIN_ORIGINS)

def summary_features(x, origins):
    origins = np.asarray(origins, dtype=np.int64)
    last = x[origins - 1]
    mean3 = np.stack([x[t-3:t].mean(axis=0) for t in origins], axis=0)
    mean12 = np.stack([x[t-12:t].mean(axis=0) for t in origins], axis=0)
    delta12 = x[origins - 1] - x[origins - 12]
    return np.stack([last, mean3, mean12, delta12], axis=-1).astype(np.float32)

def choose_targets(C):
    if C <= MAX_TARGET_CHANNELS:
        return np.arange(C, dtype=np.int64)
    return np.unique(np.linspace(0, C-1, MAX_TARGET_CHANNELS, dtype=np.int64))

def effective_topk(C):
    return int(max(1, min(MAX_TOPK, math.ceil((C - 1) / 2))))


In [5]:

def source_candidate_map(train_features, targets):
    # Same horizon-invariant candidate-pool construction as the existing controlled experiment.
    current = train_features[:, :, 0].astype(np.float64)
    current = current - current.mean(axis=0, keepdims=True)
    denom = np.maximum(np.sqrt(np.sum(current * current, axis=0)), 1e-12)
    normed = current / denom[None, :]
    C = current.shape[1]
    pool_size = min(MAX_SOURCE_CANDIDATES, max(1, C - 1))

    result = {}
    for i0 in targets:
        i = int(i0)
        corr = np.abs(normed[:, i] @ normed)
        corr[i] = -np.inf
        idx = np.argpartition(corr, -pool_size)[-pool_size:]
        idx = idx[np.argsort(corr[idx])[::-1]]
        result[i] = idx.astype(np.int64)
    return result


## 3. Exact Ridge utility used by the existing paper

In [6]:

def add_bias_column(X):
    X = np.asarray(X, dtype=np.float64)
    return np.concatenate([np.ones((len(X), 1), dtype=np.float64), X], axis=1)

def ridge_fit(X, y, lam):
    Xb = add_bias_column(X)
    A = Xb.T @ Xb
    reg = np.eye(A.shape[0], dtype=np.float64)
    reg[0, 0] = 0.0
    return np.linalg.solve(A + lam * reg, Xb.T @ np.asarray(y, dtype=np.float64))

def ridge_predict(X, beta):
    return add_bias_column(X) @ beta

def residualize_sources_against_target(Xt_fit, Xt_score, U_fit, U_score):
    # Linear residualization used only on the chronological fit prefix.
    Xt_fit_b = add_bias_column(Xt_fit)
    Xt_score_b = add_bias_column(Xt_score)

    S = U_fit.shape[1]
    Fdim = U_fit.shape[2]
    U_fit_flat = U_fit.reshape(len(U_fit), S * Fdim)

    A = Xt_fit_b.T @ Xt_fit_b
    reg = np.eye(A.shape[0], dtype=np.float64)
    reg[0, 0] = 0.0
    coef = np.linalg.solve(
        A + RIDGE_LAMBDA * reg,
        Xt_fit_b.T @ U_fit_flat,
    )

    fit_res = (U_fit_flat - Xt_fit_b @ coef).reshape(len(U_fit), S, Fdim)
    score_flat = U_score.reshape(len(U_score), S * Fdim)
    score_res = (score_flat - Xt_score_b @ coef).reshape(len(U_score), S, Fdim)
    return fit_res, score_res

def ridge_utility_for_target(X_train_all, y_train, target_idx, source_ids):
    N = X_train_all.shape[0]
    split = min(max(int(TEACHER_FIT_FRACTION * N), 32), N - 16)
    fit_idx = np.arange(split, dtype=np.int64)
    score_idx = np.arange(split, N, dtype=np.int64)

    Xt = X_train_all[:, target_idx, :].astype(np.float64)
    U = X_train_all[:, source_ids, :].astype(np.float64)
    y = np.asarray(y_train, dtype=np.float64)

    beta_t = ridge_fit(Xt[fit_idx], y[fit_idx], RIDGE_LAMBDA)
    r_fit = y[fit_idx] - ridge_predict(Xt[fit_idx], beta_t)
    r_score = y[score_idx] - ridge_predict(Xt[score_idx], beta_t)
    base_mse = float(np.mean(r_score ** 2))

    U_fit_res, U_score_res = residualize_sources_against_target(
        Xt[fit_idx], Xt[score_idx], U[fit_idx], U[score_idx]
    )

    gram = np.einsum("nsf,nsg->sfg", U_fit_res, U_fit_res)
    cross = np.einsum("nsf,n->sf", U_fit_res, r_fit)
    eye = np.eye(U_fit_res.shape[2], dtype=np.float64)[None, :, :]
    beta_s = np.linalg.solve(
        gram + SOURCE_RIDGE_LAMBDA * eye,
        cross[:, :, None],
    )[:, :, 0]

    pred_res = np.einsum("nsf,sf->ns", U_score_res, beta_s)
    err = r_score[:, None] - pred_res
    mse_source = np.mean(err ** 2, axis=0)
    utility = 100.0 * (base_mse - mse_source) / max(base_mse, 1e-12)

    return {
        "source_ids": np.asarray(source_ids, dtype=np.int64),
        "utility": utility.astype(np.float32),
        "base_mse": base_mse,
        "fit_idx": fit_idx,
        "score_idx": score_idx,
        "Xt_fit": Xt[fit_idx].astype(np.float32),
        "Xt_score": Xt[score_idx].astype(np.float32),
        "U_fit_res": U_fit_res.astype(np.float32),
        "U_score_res": U_score_res.astype(np.float32),
        "y_fit": y[fit_idx].astype(np.float32),
        "y_score": y[score_idx].astype(np.float32),
    }



## 4. Independent nonlinear utility probe

For each target–horizon condition, the MLP probe uses the **same fit/score split** as Ridge.

- Baseline MLP: target-history summary only.
- Source MLP \(j\): target-history summary + residualized source-\(j\) summary.
- Utility is measured on the untouched chronological score suffix.
- All source-specific MLPs are trained in parallel on GPU, but have **independent parameters**.
- The score suffix is never used for model selection or early stopping.

This makes the comparison a true probe-level replication rather than reusing Ridge-selected sources.


In [7]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class SmallMLP(nn.Module):
    def __init__(self, in_dim, hidden=MLP_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(MLP_DROPOUT),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

class ParallelSourceMLP(nn.Module):
    """
    S independent two-hidden-layer MLPs trained in parallel.
    Input: [B, S, D]
    Output: [B, S]
    No parameters are shared across sources.
    """
    def __init__(self, n_sources, in_dim, hidden=MLP_HIDDEN):
        super().__init__()
        S, D, H = n_sources, in_dim, hidden
        self.W1 = nn.Parameter(torch.empty(S, D, H))
        self.b1 = nn.Parameter(torch.zeros(S, H))
        self.g1 = nn.Parameter(torch.ones(S, H))
        self.be1 = nn.Parameter(torch.zeros(S, H))
        self.W2 = nn.Parameter(torch.empty(S, H, H))
        self.b2 = nn.Parameter(torch.zeros(S, H))
        self.W3 = nn.Parameter(torch.empty(S, H))
        self.b3 = nn.Parameter(torch.zeros(S))
        self.reset_parameters()

    def reset_parameters(self):
        for s in range(self.W1.shape[0]):
            nn.init.kaiming_uniform_(self.W1[s], a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.W2[s], a=math.sqrt(5))
            nn.init.uniform_(self.W3[s], -1/math.sqrt(self.W3.shape[1]), 1/math.sqrt(self.W3.shape[1]))

    def forward(self, x):
        h = torch.einsum("bsd,sdh->bsh", x, self.W1) + self.b1[None, :, :]
        mean = h.mean(dim=-1, keepdim=True)
        var = h.var(dim=-1, unbiased=False, keepdim=True)
        h = (h - mean) / torch.sqrt(var + 1e-5)
        h = h * self.g1[None, :, :] + self.be1[None, :, :]
        h = F.gelu(h)
        h = F.dropout(h, p=MLP_DROPOUT, training=self.training)
        h = torch.einsum("bsh,shk->bsk", h, self.W2) + self.b2[None, :, :]
        h = F.gelu(h)
        return torch.einsum("bsh,sh->bs", h, self.W3) + self.b3[None, :]

def standardize_2d_fit_score(Xfit, Xscore):
    mu = Xfit.mean(axis=0, keepdims=True)
    sd = np.maximum(Xfit.std(axis=0, keepdims=True), 1e-6)
    return (Xfit-mu)/sd, (Xscore-mu)/sd

def fit_base_mlp_score_mse(Xfit, yfit, Xscore, yscore, seed):
    set_seed(seed)
    Xfit, Xscore = standardize_2d_fit_score(Xfit.astype(np.float32), Xscore.astype(np.float32))
    yfit = np.asarray(yfit, dtype=np.float32)
    yscore = np.asarray(yscore, dtype=np.float32)

    model = SmallMLP(Xfit.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    rng = np.random.default_rng(seed)

    for _ in range(MLP_EPOCHS):
        model.train()
        order = rng.permutation(len(Xfit))
        for st in range(0, len(order), MLP_BATCH_SIZE):
            idx = order[st:st+MLP_BATCH_SIZE]
            xb = torch.from_numpy(Xfit[idx]).to(DEVICE)
            yb = torch.from_numpy(yfit[idx]).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            opt.step()

    model.eval()
    preds = []
    with torch.inference_mode():
        for st in range(0, len(Xscore), 1024):
            xb = torch.from_numpy(Xscore[st:st+1024]).to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    pred = np.concatenate(preds)
    mse = float(np.mean((yscore - pred) ** 2))
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return mse

def fit_parallel_source_mlps_score_mse(Xt_fit, U_fit_res, yfit, Xt_score, U_score_res, yscore, seed):
    set_seed(seed)
    Xt_fit = np.asarray(Xt_fit, dtype=np.float32)
    Xt_score = np.asarray(Xt_score, dtype=np.float32)
    U_fit_res = np.asarray(U_fit_res, dtype=np.float32)
    U_score_res = np.asarray(U_score_res, dtype=np.float32)
    yfit = np.asarray(yfit, dtype=np.float32)
    yscore = np.asarray(yscore, dtype=np.float32)

    # [N,S,Dt+Ds]
    Xt_fit_rep = np.repeat(Xt_fit[:, None, :], U_fit_res.shape[1], axis=1)
    Xt_score_rep = np.repeat(Xt_score[:, None, :], U_score_res.shape[1], axis=1)
    Xfit = np.concatenate([Xt_fit_rep, U_fit_res], axis=2)
    Xscore = np.concatenate([Xt_score_rep, U_score_res], axis=2)

    # Standardization is fit-only and source-specific.
    mu = Xfit.mean(axis=0, keepdims=True)
    sd = np.maximum(Xfit.std(axis=0, keepdims=True), 1e-6)
    Xfit = ((Xfit - mu) / sd).astype(np.float32)
    Xscore = ((Xscore - mu) / sd).astype(np.float32)

    S = Xfit.shape[1]
    model = ParallelSourceMLP(S, Xfit.shape[2]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WEIGHT_DECAY)
    rng = np.random.default_rng(seed)

    for _ in range(MLP_EPOCHS):
        model.train()
        order = rng.permutation(len(Xfit))
        for st in range(0, len(order), MLP_BATCH_SIZE):
            idx = order[st:st+MLP_BATCH_SIZE]
            xb = torch.from_numpy(Xfit[idx]).to(DEVICE)
            yb = torch.from_numpy(yfit[idx]).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            # Sum of per-source mean losses => each source receives its own independent MSE gradient.
            per_source_loss = ((pred - yb[:, None]) ** 2).mean(dim=0)
            loss = per_source_loss.sum()
            loss.backward()
            opt.step()

    model.eval()
    sqerr_sum = np.zeros(S, dtype=np.float64)
    n = 0
    with torch.inference_mode():
        for st in range(0, len(Xscore), 512):
            xb = torch.from_numpy(Xscore[st:st+512]).to(DEVICE)
            pred = model(xb).cpu().numpy()
            yy = yscore[st:st+len(pred), None]
            sqerr_sum += ((pred - yy) ** 2).sum(axis=0)
            n += len(pred)

    mse = (sqerr_sum / max(n, 1)).astype(np.float32)
    del model, Xfit, Xscore, Xt_fit_rep, Xt_score_rep
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return mse

def mlp_utility_from_ridge_cache(ridge_cache, seeds):
    base_mses = []
    source_mses = []
    for seed in seeds:
        base = fit_base_mlp_score_mse(
            ridge_cache["Xt_fit"], ridge_cache["y_fit"],
            ridge_cache["Xt_score"], ridge_cache["y_score"],
            seed,
        )
        smse = fit_parallel_source_mlps_score_mse(
            ridge_cache["Xt_fit"], ridge_cache["U_fit_res"], ridge_cache["y_fit"],
            ridge_cache["Xt_score"], ridge_cache["U_score_res"], ridge_cache["y_score"],
            seed,
        )
        base_mses.append(base)
        source_mses.append(smse)

    base_mses = np.asarray(base_mses, dtype=np.float64)
    source_mses = np.stack(source_mses, axis=0).astype(np.float64)
    utility_by_seed = 100.0 * (base_mses[:, None] - source_mses) / np.maximum(base_mses[:, None], 1e-12)

    return {
        "utility_mean": utility_by_seed.mean(axis=0).astype(np.float32),
        "utility_std": utility_by_seed.std(axis=0).astype(np.float32),
        "base_mse_mean": float(base_mses.mean()),
        "base_mse_std": float(base_mses.std()),
    }


## 5. Optional EPI-pair augmentation

In [8]:

if EPI_PAIR_FILE.exists():
    epi_pairs = pd.read_csv(EPI_PAIR_FILE)
    needed = {"dataset", "pred_len", "target_channel", "source_channel"}
    if not needed.issubset(epi_pairs.columns):
        raise RuntimeError(f"EPI file missing columns: {needed - set(epi_pairs.columns)}")
    print("Loaded EPI rows:", len(epi_pairs))
    display(epi_pairs.head())
else:
    epi_pairs = pd.DataFrame()
    print("EPI file not found. Cross-probe General-20 analysis can still run; EPI alignment will be skipped.")

def matched_epi_sources(dataset, H, target):
    if len(epi_pairs) == 0:
        return np.array([], dtype=np.int64)
    sub = epi_pairs[
        (epi_pairs["dataset"] == dataset)
        & (epi_pairs["pred_len"].astype(int) == int(H))
        & (epi_pairs["target_channel"].astype(int) == int(target))
    ]
    return np.unique(sub["source_channel"].astype(int).to_numpy())


Loaded EPI rows: 737


,dataset,pred_len,target_channel,source_channel,raw_abs_corr,latent_cosine,latent_abs_cosine,predictive_utility_%,neural_full_importance_%,neural_endpoint_importance_%,neural_full_importance_seed_std,neural_endpoint_importance_seed_std,candidate_count,test_windows
0,Electricity,192,0,123,0.203406,0.226049,0.239185,1.676855,0.077705,-0.038038,0.798245,0.599941,18,256
1,Electricity,192,0,99,0.194923,0.290626,0.296842,0.545547,-0.049840,-0.813900,0.549488,0.624531,18,256
2,Electricity,192,0,46,0.194802,0.355256,0.358629,-6.502506,-0.136587,-0.382846,0.186884,0.023323,18,256
3,Electricity,192,0,128,0.178045,0.360931,0.365639,-2.658021,0.137141,0.109292,0.039176,0.414290,18,256
4,Electricity,192,0,127,0.176957,0.292320,0.301112,0.409886,0.062196,0.125349,0.280753,0.166636,18,256


## 6. Run experiment

In [9]:

conditions = SCREEN_CONDITIONS if RUN_MODE == "screen" else FULL_CONDITIONS
mlp_seeds = SCREEN_MLP_SEEDS if RUN_MODE == "screen" else FULL_MLP_SEEDS

print("Conditions:", len(conditions))
print("MLP seeds:", mlp_seeds)

def run_dataset(dataset_name, requested_horizons):
    spec = DATASET_REGISTRY[dataset_name]
    path = resolve_path(spec["paths"])
    if path is None:
        raise FileNotFoundError(dataset_name)

    raw = load_series(dataset_name, path)
    T, C = raw.shape
    train_end, val_end, test_end = split_boundaries(dataset_name, T)
    x = standardize_train_only(raw, train_end)

    train_origins = make_train_origins(train_end)
    X_train = summary_features(x, train_origins)
    targets = choose_targets(C)
    candidate_map = source_candidate_map(X_train, targets)
    K = effective_topk(C)

    rows = []
    t0 = time.time()

    for ti, target0 in enumerate(targets):
        i = int(target0)

        for H in sorted(set(requested_horizons)):
            # Start from the paper's controlled candidate pool.
            source_ids = candidate_map[i].copy()

            # Ensure every source used by the matched EPI analysis is also evaluated by the MLP probe.
            extra = matched_epi_sources(dataset_name, H, i)
            if len(extra):
                source_ids = np.unique(np.concatenate([source_ids, extra]))
                source_ids = source_ids[source_ids != i]

            y_train = x[train_origins + int(H) - 1, i]

            ridge = ridge_utility_for_target(
                X_train, y_train, i, source_ids
            )
            mlp = mlp_utility_from_ridge_cache(ridge, mlp_seeds)

            for s_idx, j in enumerate(source_ids):
                rows.append({
                    "dataset": dataset_name,
                    "horizon": int(H),
                    "target_channel": i,
                    "source_channel": int(j),
                    "C": int(C),
                    "K": int(K),
                    "in_controlled_candidate_pool": int(j in set(candidate_map[i].tolist())),
                    "in_epi_pair_pool": int(j in set(extra.tolist())),
                    "ridge_utility_%": float(ridge["utility"][s_idx]),
                    "mlp_utility_mean_%": float(mlp["utility_mean"][s_idx]),
                    "mlp_utility_std_%": float(mlp["utility_std"][s_idx]),
                    "mlp_base_mse_mean": float(mlp["base_mse_mean"]),
                    "mlp_base_mse_std": float(mlp["base_mse_std"]),
                })

        if (ti + 1) % max(1, len(targets)//4) == 0:
            print(f"  {dataset_name}: {ti+1}/{len(targets)} targets | elapsed {(time.time()-t0)/60:.1f} min")

    return pd.DataFrame(rows)

by_dataset = {}
for d, h in conditions:
    by_dataset.setdefault(d, []).append(int(h))

frames, failures = [], []

for dataset, hs in by_dataset.items():
    out_csv = OUTPUT_DIR / f"{RUN_MODE}_{dataset}_pair_utilities.csv"
    if out_csv.exists():
        print("Reusing:", out_csv)
        frames.append(pd.read_csv(out_csv))
        continue

    print("\n" + "="*90)
    print(dataset, sorted(set(hs)))
    print("="*90)

    try:
        df = run_dataset(dataset, hs)
        df.to_csv(out_csv, index=False)
        frames.append(df)
    except Exception as e:
        print("FAILED:", dataset, repr(e))
        failures.append({"dataset": dataset, "error": repr(e)})

if not frames:
    raise RuntimeError("No completed dataset.")

pair_rows = pd.concat(frames, ignore_index=True)
pair_rows.to_csv(OUTPUT_DIR / f"{RUN_MODE}_pair_utilities_all.csv", index=False)
pd.DataFrame(failures).to_csv(OUTPUT_DIR / f"{RUN_MODE}_failures.csv", index=False)

print("\nRows:", len(pair_rows), "| failures:", len(failures))
display(pair_rows.head())


Conditions: 8
MLP seeds: [2026]
Reusing: /data/code/2026_08/results_multiprobe_predictive_utility/screen_Electricity_pair_utilities.csv
Reusing: /data/code/2026_08/results_multiprobe_predictive_utility/screen_Solar_pair_utilities.csv
Reusing: /data/code/2026_08/results_multiprobe_predictive_utility/screen_Weather_pair_utilities.csv
Reusing: /data/code/2026_08/results_multiprobe_predictive_utility/screen_ETTh1_pair_utilities.csv
Reusing: /data/code/2026_08/results_multiprobe_predictive_utility/screen_ETTm1_pair_utilities.csv

Rows: 16958 | failures: 0


,dataset,horizon,target_channel,source_channel,C,K,in_controlled_candidate_pool,in_epi_pair_pool,ridge_utility_%,mlp_utility_mean_%,mlp_utility_std_%,mlp_base_mse_mean,mlp_base_mse_std
0,Electricity,192,0,1,321,10,1,0,0.271373,2.815430,0.0,1.19059,0.0
1,Electricity,192,0,2,321,10,1,0,-0.972755,1.067886,0.0,1.19059,0.0
2,Electricity,192,0,8,321,10,1,0,-0.009435,0.000451,0.0,1.19059,0.0
3,Electricity,192,0,9,321,10,1,0,0.230337,-1.561058,0.0,1.19059,0.0
4,Electricity,192,0,11,321,10,1,0,-0.481995,0.194585,0.0,1.19059,0.0


## 7. Cross-probe ranking agreement

In [10]:

def safe_spearman(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return np.nan
    a, b = a[ok], b[ok]
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    res = spearmanr(a, b)
    # SciPy compatibility:
    # - newer versions may expose `.statistic`
    # - older versions expose `.correlation`
    # - tuple-style access [0] works as a final fallback
    if hasattr(res, "statistic"):
        return float(res.statistic)
    if hasattr(res, "correlation"):
        return float(res.correlation)
    return float(res[0])

def top_ids(df, score_col, k):
    s = df.sort_values(score_col, ascending=False)
    return set(s["source_channel"].astype(int).head(min(k, len(s))).tolist())

def set_jaccard(a, b):
    u = a | b
    return 1.0 if not u else len(a & b) / len(u)

def recall_a_in_b(a, b):
    return np.nan if not a else len(a & b) / len(a)

# Primary cross-probe comparison is restricted to the exact controlled candidate pool,
# not the EPI-only augmented sources.
primary = pair_rows[pair_rows["in_controlled_candidate_pool"] == 1].copy()

target_rows = []
for (d, H, t), sub in primary.groupby(["dataset", "horizon", "target_channel"]):
    K = int(sub["K"].iloc[0])
    r_top = top_ids(sub, "ridge_utility_%", K)
    m_top = top_ids(sub, "mlp_utility_mean_%", K)
    target_rows.append({
        "dataset": d,
        "horizon": int(H),
        "target_channel": int(t),
        "n_sources": len(sub),
        "K": K,
        "rho_ridge_mlp": safe_spearman(sub["ridge_utility_%"], sub["mlp_utility_mean_%"]),
        "topk_jaccard_ridge_mlp": set_jaccard(r_top, m_top),
        "ridge_topk_recall_in_mlp": recall_a_in_b(r_top, m_top),
        "mlp_topk_recall_in_ridge": recall_a_in_b(m_top, r_top),
    })

target_agreement = pd.DataFrame(target_rows)
target_agreement.to_csv(OUTPUT_DIR / f"{RUN_MODE}_target_crossprobe_agreement.csv", index=False)

condition_agreement = (
    target_agreement
    .groupby(["dataset", "horizon"], as_index=False)
    .agg(
        n_targets=("target_channel", "size"),
        median_rho_ridge_mlp=("rho_ridge_mlp", "median"),
        mean_topk_jaccard=("topk_jaccard_ridge_mlp", "mean"),
        mean_ridge_recall=("ridge_topk_recall_in_mlp", "mean"),
    )
)
condition_agreement.to_csv(OUTPUT_DIR / f"{RUN_MODE}_condition_crossprobe_agreement.csv", index=False)

display(condition_agreement.round(3))

print("\nGLOBAL CROSS-PROBE AGREEMENT")
print("Target instances:", len(target_agreement))
print("Median Spearman:", round(float(target_agreement["rho_ridge_mlp"].median()), 3))
print("Mean Top-K Jaccard:", round(float(target_agreement["topk_jaccard_ridge_mlp"].mean()), 3))
print("Mean Ridge-TopK recall in MLP:", round(float(target_agreement["ridge_topk_recall_in_mlp"].mean()), 3))


,dataset,horizon,n_targets,median_rho_ridge_mlp,mean_topk_jaccard,mean_ridge_recall
0,ETTh1,336,7,-0.029,0.371,0.476
1,ETTh1,720,7,0.429,0.429,0.571
2,ETTm1,720,7,0.371,0.557,0.667
3,Electricity,192,32,0.319,0.161,0.266
4,Electricity,336,32,0.281,0.132,0.219
5,Solar,336,32,0.439,0.138,0.225
6,Solar,720,32,0.532,0.206,0.316
7,Weather,720,21,0.032,0.353,0.505



GLOBAL CROSS-PROBE AGREEMENT
Target instances: 170
Median Spearman: 0.365
Mean Top-K Jaccard: 0.219
Mean Ridge-TopK recall in MLP: 0.326


## 8. Matched comparison to existing iTransformer EPI

In [11]:

epi_target_alignment = pd.DataFrame()

if len(epi_pairs):
    # Use only exact pairs that already have neural reliance measured.
    keep_cols = [
        "dataset", "pred_len", "target_channel", "source_channel",
        "neural_endpoint_importance_%",
    ]
    if "predictive_utility_%" in epi_pairs.columns:
        keep_cols.append("predictive_utility_%")

    epi = epi_pairs[keep_cols].copy()
    epi = epi.rename(columns={"pred_len": "horizon"})

    merged = epi.merge(
        pair_rows,
        on=["dataset", "horizon", "target_channel", "source_channel"],
        how="inner",
        validate="one_to_one",
    )
    merged.to_csv(OUTPUT_DIR / f"{RUN_MODE}_matched_epi_pairs.csv", index=False)

    rows = []
    for (d, H, t), sub in merged.groupby(["dataset", "horizon", "target_channel"]):
        rows.append({
            "dataset": d,
            "horizon": int(H),
            "target_channel": int(t),
            "n_pairs": len(sub),
            "rho_ridge_epi_recomputed": safe_spearman(
                sub["ridge_utility_%"], sub["neural_endpoint_importance_%"]
            ),
            "rho_mlp_epi": safe_spearman(
                sub["mlp_utility_mean_%"], sub["neural_endpoint_importance_%"]
            ),
            "rho_ridge_mlp_on_epi_pairs": safe_spearman(
                sub["ridge_utility_%"], sub["mlp_utility_mean_%"]
            ),
        })

        if "predictive_utility_%" in sub.columns:
            rows[-1]["rho_originalridge_epi"] = safe_spearman(
                sub["predictive_utility_%"], sub["neural_endpoint_importance_%"]
            )
            rows[-1]["rho_originalridge_recomputedridge"] = safe_spearman(
                sub["predictive_utility_%"], sub["ridge_utility_%"]
            )

    epi_target_alignment = pd.DataFrame(rows)
    epi_target_alignment.to_csv(
        OUTPUT_DIR / f"{RUN_MODE}_epi_target_alignment.csv", index=False
    )

    display(epi_target_alignment.round(3))

    print("\nMATCHED EPI ALIGNMENT")
    print("Target instances:", len(epi_target_alignment))
    print("Median Ridge–EPI:",
          round(float(epi_target_alignment["rho_ridge_epi_recomputed"].median()), 3))
    print("Median MLP–EPI:",
          round(float(epi_target_alignment["rho_mlp_epi"].median()), 3))
    print("Median Ridge–MLP on EPI pairs:",
          round(float(epi_target_alignment["rho_ridge_mlp_on_epi_pairs"].median()), 3))
    if "rho_originalridge_recomputedridge" in epi_target_alignment.columns:
        print("Median original-vs-recomputed Ridge fidelity:",
              round(float(epi_target_alignment["rho_originalridge_recomputedridge"].median()), 3))
else:
    print("Skipped because the existing EPI pair-level file was not found.")


,dataset,horizon,target_channel,n_pairs,rho_ridge_epi_recomputed,rho_mlp_epi,rho_ridge_mlp_on_epi_pairs,rho_originalridge_epi,rho_originalridge_recomputedridge
0,ETTh1,336,0,6,-0.257,0.371,-0.486,-0.429,0.886
1,ETTh1,336,1,6,-0.029,-0.600,-0.029,-0.029,1.000
2,ETTh1,336,2,6,-0.143,0.429,-0.714,-0.371,0.943
3,ETTh1,336,3,6,0.486,0.143,0.543,0.486,1.000
4,ETTh1,336,4,6,0.886,0.714,0.829,0.943,0.943
5,ETTh1,336,5,6,-0.600,-0.829,0.486,-0.543,0.943
6,ETTh1,336,6,6,0.771,-0.943,-0.829,0.771,1.000
7,ETTh1,720,0,6,-0.657,0.086,0.543,-0.429,0.943
8,ETTh1,720,1,6,0.143,0.029,-0.657,0.086,0.943
9,ETTh1,720,2,6,-0.600,0.257,0.371,-0.371,0.943



MATCHED EPI ALIGNMENT
Target instances: 39
Median Ridge–EPI: 0.004
Median MLP–EPI: -0.143
Median Ridge–MLP on EPI pairs: 0.324
Median original-vs-recomputed Ridge fidelity: 0.943


## 9. Fixed interpretation guide

In [12]:

print("=" * 92)
print("EXP4 FIXED INTERPRETATION CRITERIA")
print("=" * 92)

rho_rm = float(target_agreement["rho_ridge_mlp"].median())
jac_rm = float(target_agreement["topk_jaccard_ridge_mlp"].mean())

print(f"Cross-probe median Spearman (Ridge vs MLP): {rho_rm:+.3f}")
print(f"Cross-probe mean Top-K Jaccard            : {jac_rm:.3f}")

if len(epi_target_alignment):
    rho_re = float(epi_target_alignment["rho_ridge_epi_recomputed"].median())
    rho_me = float(epi_target_alignment["rho_mlp_epi"].median())
    print(f"Matched median Ridge vs EPI               : {rho_re:+.3f}")
    print(f"Matched median MLP   vs EPI               : {rho_me:+.3f}")

print("\nInterpretation:")
print("A) Ridge–MLP high, both EPI alignments weak:")
print("   strongest support that controlled utility is not a Ridge-only artifact,")
print("   while utility remains distinct from neural functional reliance.")
print("B) Ridge–MLP low:")
print("   report predictive utility as explicitly probe-conditioned; this is still")
print("   scientifically informative and requires narrowing, not hiding, the claim.")
print("C) MLP–EPI becomes strong:")
print("   neural reliance may be better explained by nonlinear utility; revise the")
print("   paper rather than presenting the mismatch as universal.")

print("\nFiles to return for analysis:")
for name in [
    f"{RUN_MODE}_pair_utilities_all.csv",
    f"{RUN_MODE}_target_crossprobe_agreement.csv",
    f"{RUN_MODE}_condition_crossprobe_agreement.csv",
    f"{RUN_MODE}_matched_epi_pairs.csv",
    f"{RUN_MODE}_epi_target_alignment.csv",
    f"{RUN_MODE}_failures.csv",
]:
    p = OUTPUT_DIR / name
    if p.exists():
        print(" -", p)

if RUN_MODE == "screen":
    print("\nNEXT: if the notebook ran correctly, set RUN_MODE='full' and rerun.")
    print("The full run is required regardless of whether the screen result is favorable.")


EXP4 FIXED INTERPRETATION CRITERIA
Cross-probe median Spearman (Ridge vs MLP): +0.365
Cross-probe mean Top-K Jaccard            : 0.219
Matched median Ridge vs EPI               : +0.004
Matched median MLP   vs EPI               : -0.143

Interpretation:
A) Ridge–MLP high, both EPI alignments weak:
   strongest support that controlled utility is not a Ridge-only artifact,
   while utility remains distinct from neural functional reliance.
B) Ridge–MLP low:
   report predictive utility as explicitly probe-conditioned; this is still
   scientifically informative and requires narrowing, not hiding, the claim.
C) MLP–EPI becomes strong:
   neural reliance may be better explained by nonlinear utility; revise the
   paper rather than presenting the mismatch as universal.

Files to return for analysis:
 - /data/code/2026_08/results_multiprobe_predictive_utility/screen_pair_utilities_all.csv
 - /data/code/2026_08/results_multiprobe_predictive_utility/screen_target_crossprobe_agreement.csv
 - /d